# 모델 학습 (train)

In [ ]:
# ========== 0. Config ==========
import os, json, random, itertools, ast
from pathlib import Path

ZIP_PATH   = "/content/drive/MyDrive/snuaichallenge.zip"   # ★ 본인 경로 ★
DRIVE_ROOT = "/content/drive/MyDrive/snuai_runs_inprogress"

CFG = dict(
    MODEL_ID   = "Qwen/Qwen3-VL-8B-Instruct",
    SEED       = 42,
    RUN_NAME   = "qwen3vl8b_v5_r64_paug4_px320",

    # 증강/TTA
    K_AUG      = 4,      # 학습 샘플당 입력 순열 수 (identity 1 + 랜덤 3)
    K_TTA      = 2,      # 추론 TTA 순열 수

    # 해상도 (기존 규약: 토큰 수 × 28×28)
    MIN_PIXELS = 128 * 28 * 28,
    MAX_PIXELS = 320 * 28 * 28,   # 기존 v4에서 병목 진단 후 확정한 값 유지

    # 학습
    LORA_R = 64, LORA_ALPHA = 128,
    LR = 1e-4, VIT_LR = 2e-5,
    EPOCHS = 1,
    BATCH = 1, GRAD_ACC = 16,

    VAL_RATIO  = 0.05,   # 기존 train.py와 동일
    LONG_WC    = 30,     # 테스트 프록시 서브셋 기준
)

random.seed(CFG["SEED"])
WORK = "/content/work"; os.makedirs(WORK, exist_ok=True)
RUN_DIR = os.path.join(DRIVE_ROOT, CFG["RUN_NAME"])
SMOKE_TEST = False      # 최초 실행은 True 권장
print(RUN_DIR)


/content/drive/MyDrive/snuai_runs_inprogress/qwen3vl8b_v5_r64_paug4_px320


In [ ]:
# ========== 1. Drive 마운트 + 데이터 압축 해제 (기존과 동일) ==========
import glob
from google.colab import drive
drive.mount('/content/drive')
assert os.path.exists(ZIP_PATH), f"zip 파일이 없습니다: {ZIP_PATH}"

!unzip -q -n "{ZIP_PATH}" -d /content/snuai_data
hits = glob.glob('/content/snuai_data/**/train.csv', recursive=True)
assert hits, "train.csv를 찾지 못했습니다."
DATA_DIR = os.path.dirname(hits[0])
print("DATA_DIR =", DATA_DIR)
for req in ["train.csv", "test.csv", "train", "test"]:
    p = os.path.join(DATA_DIR, req)
    print(" ", "OK " if os.path.exists(p) else "MISSING!!", p)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATA_DIR = /content/snuai_data/snuaichallenge_data
  OK  /content/snuai_data/snuaichallenge_data/train.csv
  OK  /content/snuai_data/snuaichallenge_data/test.csv
  OK  /content/snuai_data/snuaichallenge_data/train
  OK  /content/snuai_data/snuaichallenge_data/test


In [ ]:
# ========== 2. 설치 (검증된 호환 버전) ==========
%pip install -q -U "ms-swift==4.4.2" "transformers==5.12.1" "datasets==4.8.4" "qwen-vl-utils" accelerate peft

import torch, transformers, datasets
from datasets.features import Json, List
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("datasets Json/List import OK")
print("GPU:", torch.cuda.get_device_name(0))


torch 2.11.0+cu128 | transformers 5.12.1 | datasets 4.8.4
datasets Json/List import OK
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
import pandas as pd

def inverse_perm(p):
    """1-indexed 순열의 역순열. Answer <-> chrono 양방향 변환 (기존 common.py와 동일)."""
    inv = [0] * len(p)
    for i, v in enumerate(p):
        inv[v - 1] = i + 1
    return inv

def image_paths_for_row(row, image_dir):
    return [os.path.join(image_dir, row["Id"], row[f"Input_{k}"]) for k in range(1, 5)]

def load_split(split):
    df = pd.read_csv(os.path.join(DATA_DIR, f"{split}.csv"))
    image_dir = os.path.join(DATA_DIR, split)
    samples = []
    for _, row in df.iterrows():
        gt = None
        if "Answer" in df.columns:
            answer = ast.literal_eval(row["Answer"])
            gt = inverse_perm(answer)          # chrono 포맷으로 변환
        samples.append(dict(id=row["Id"], caption=str(row["Sentence"]),
                            frames=image_paths_for_row(row, image_dir), gt=gt))
    return samples

all_train = load_split("train")
test_set  = load_split("test")

# 기존 train.py와 동일한 split: seed 42 셔플 -> 앞 5% = val
df_idx = pd.Series(range(len(all_train))).sample(frac=1.0, random_state=CFG["SEED"]).tolist()
n_val = int(len(all_train) * CFG["VAL_RATIO"])
val_set   = [all_train[i] for i in df_idx[:n_val]]
train_set = [all_train[i] for i in df_idx[n_val:]]
print(f"train {len(train_set)} / val {len(val_set)} / test {len(test_set)}")

if SMOKE_TEST:
    train_set, val_set, test_set = train_set[:16], val_set[:4], test_set[:4]
    print("SMOKE_TEST: 극소량으로 축소")


train 9059 / val 476 / test 819


In [ ]:
USER_TEXT_TMPL = (
    'Story: "{sentence}"\n'
    "The 4 images above (Image 1 to Image 4) are shuffled frames from a single video. "
    "Reorder them into the correct chronological order so that they match the story. "
    "If the story states an explicit sequence (e.g. words like 'then', 'after', "
    "'finally', 'begins by'), use that order directly. "
    "If the story does NOT state an explicit sequence, it describes a single continuous "
    "action or scene: compare the images for visual progression cues such as body/object "
    "position, motion direction, pose changes, or accumulated change (e.g. water level, "
    "object displacement, distance covered), and order the frames from earliest to latest "
    "based on those visual cues. "
    "Answer ONLY with a Python list of the image numbers in chronological order. "
    "Example: [3, 1, 4, 2]"
)

def swift_user_content(sentence):
    # 기존 build_messages의 인터리브 구조를 ms-swift 텍스트 포맷으로: <image>\nImage N\n ...
    parts = []
    for i in range(4):
        parts.append("<image>")
        parts.append(f"\nImage {i + 1}\n")
    parts.append(USER_TEXT_TMPL.format(sentence=sentence))
    return "".join(parts)

def format_chrono(chrono):
    return str(list(chrono))     # '[3, 1, 4, 2]' — 기존과 동일

def apply_perm(sample, sigma):
    frames = [sample["frames"][sigma[j] - 1] for j in range(4)]
    gt = None
    if sample["gt"] is not None:
        iv = inverse_perm(sigma)
        gt = [iv[i - 1] for i in sample["gt"]]
    return frames, gt

def sample_perms(k, rng):
    perms = [(1, 2, 3, 4)]
    pool = [p for p in itertools.permutations([1, 2, 3, 4]) if p != (1, 2, 3, 4)]
    perms += rng.sample(pool, min(k - 1, len(pool)))
    return [list(p) for p in perms]

def build_jsonl(samples, path, k_aug, seed):
    rng = random.Random(seed)
    n = 0
    with open(path, "w") as f:
        for s in samples:
            for sigma in sample_perms(k_aug, rng):
                frames, gt = apply_perm(s, sigma)
                rec = {"messages": [
                          {"role": "user", "content": swift_user_content(s["caption"])},
                          {"role": "assistant", "content": format_chrono(gt)}],
                       "images": frames}
                f.write(json.dumps(rec, ensure_ascii=False) + "\n"); n += 1
    print(path, n, "rows")

TRAIN_JSONL = os.path.join(WORK, "train_aug.jsonl")
VAL_JSONL   = os.path.join(WORK, "val_plain.jsonl")
build_jsonl(train_set, TRAIN_JSONL, CFG["K_AUG"], CFG["SEED"])
build_jsonl(val_set,   VAL_JSONL,   1,            CFG["SEED"])


/content/work/train_aug.jsonl 36236 rows
/content/work/val_plain.jsonl 476 rows


In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
# ========== 5. 학습 (SDPA fallback) ==========
import os

os.environ["MIN_PIXELS"] = str(CFG["MIN_PIXELS"])
os.environ["MAX_PIXELS"] = str(CFG["MAX_PIXELS"])

# 현재 ms-swift용 이미지 토큰 설정
os.environ["IMAGE_MIN_TOKEN_NUM"] = "128"
os.environ["IMAGE_MAX_TOKEN_NUM"] = "320"

OUT = os.path.join(WORK, "out", CFG["RUN_NAME"])

!swift sft \
  --model {CFG["MODEL_ID"]} \
  --dataset {TRAIN_JSONL} \
  --val_dataset {VAL_JSONL} \
  --tuner_type lora \
  --lora_rank {CFG["LORA_R"]} \
  --lora_alpha {CFG["LORA_ALPHA"]} \
  --lora_dropout 0.05 \
  --target_modules all-linear \
  --freeze_vit false \
  --freeze_aligner false \
  --learning_rate {CFG["LR"]} \
  --vit_lr {CFG["VIT_LR"]} \
  --num_train_epochs {CFG["EPOCHS"]} \
  --per_device_train_batch_size {CFG["BATCH"]} \
  --gradient_accumulation_steps {CFG["GRAD_ACC"]} \
  --packing false \
  --torch_dtype bfloat16 \
  --attn_impl sdpa \
  --warmup_ratio 0.03 \
  --lr_scheduler_type cosine \
  --gradient_checkpointing true \
  --save_steps 200 \
  --save_total_limit 3 \
  --logging_steps 10 \
  --max_length 4096 \
  --output_dir {OUT} \
  --seed {CFG["SEED"]}

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model Qwen/Qwen3-VL-8B-Instruct --dataset /content/work/train_aug.jsonl --val_dataset /content/work/val_plain.jsonl --tuner_type lora --lora_rank 64 --lora_alpha 128 --lora_dropout 0.05 --target_modules all-linear --freeze_vit false --freeze_aligner false --learning_rate 0.0001 --vit_lr 2e-05 --num_train_epochs 1 --per_device_train_batch_size 1 --gradient_accumulation_steps 16 --packing false --torch_dtype bfloat16 --attn_impl sdpa --warmup_ratio 0.03 --lr_scheduler_type cosine --gradient_checkpointing true --save_steps 200 --save_total_limit 3 --logging_steps 10 --max_length 4096 --output_dir /content/work/out/qwen3vl8b_v5_r64_paug4_px320 --seed 42`
[INFO:swift] Successfully registered `/usr/local/lib/python3.12/dist-packages/swift/dataset/data/dataset_info.json`.
[INFO:swift] rank: -1, local_rank: -1, world_size: 1, local_world_size: 1
[INFO:swift] Downloading the model from ModelScope Hub, model_id:

In [ ]:
# ========== 5b. Drive 백업 ==========
import os, glob, shutil, subprocess

assert os.path.isdir("/content/drive/MyDrive"), "Drive가 마운트되지 않았습니다."
assert RUN_DIR.startswith("/content/drive/"), f"RUN_DIR이 Drive 경로가 아님: {RUN_DIR}"
os.makedirs(RUN_DIR, exist_ok=True)

ckpts = sorted(
    (p for p in glob.glob(os.path.join(OUT, "**", "checkpoint-*"), recursive=True)
     if os.path.isdir(p)),
    key=lambda p: int(p.rsplit("-", 1)[1]),
)
assert ckpts, "checkpoint가 없습니다."

print("발견:", [os.path.basename(c) for c in ckpts])
subprocess.run(["du", "-sh", *ckpts])

for src in ckpts:
    ver = os.path.basename(os.path.dirname(src))          # v1-20260723-090258
    dst = os.path.join(RUN_DIR, ver, os.path.basename(src))
    os.makedirs(dst, exist_ok=True)
    subprocess.run(["rsync", "-a", f"{src}/", f"{dst}/"])  # 기존 파일은 건너뜀
    print("완료:", dst)

for f in glob.glob(os.path.join(OUT, "*", "logging.jsonl")) + \
         glob.glob(os.path.join(OUT, "*", "args.json")):
    ver = os.path.basename(os.path.dirname(f))
    os.makedirs(os.path.join(RUN_DIR, ver), exist_ok=True)
    shutil.copy2(f, os.path.join(RUN_DIR, ver, os.path.basename(f)))
print("로그/설정 백업 완료")

발견: ['checkpoint-1000', 'checkpoint-2200', 'checkpoint-2265']
완료: /content/drive/MyDrive/snuai_runs_inprogress/qwen3vl8b_v5_r64_paug4_px320/v1-20260723-090258/checkpoint-1000
완료: /content/drive/MyDrive/snuai_runs_inprogress/qwen3vl8b_v5_r64_paug4_px320/v1-20260723-090258/checkpoint-2200
완료: /content/drive/MyDrive/snuai_runs_inprogress/qwen3vl8b_v5_r64_paug4_px320/v1-20260723-090258/checkpoint-2265
로그/설정 백업 완료


In [ ]:
import torch, itertools, random, zlib
import pandas as pd
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel

BASE = "/root/.cache/modelscope/models/Qwen--Qwen3-VL-8B-Instruct/snapshots/master"
CKPT = ckpts[-1]
device = "cuda"

try:
    processor = AutoProcessor.from_pretrained(
        CKPT, min_pixels=CFG["MIN_PIXELS"], max_pixels=CFG["MAX_PIXELS"])
except Exception as e:
    print("체크포인트에 processor 없음, base 사용:", e)
    processor = AutoProcessor.from_pretrained(
        BASE, min_pixels=CFG["MIN_PIXELS"], max_pixels=CFG["MAX_PIXELS"])

base = AutoModelForImageTextToText.from_pretrained(
    BASE, dtype=torch.bfloat16, device_map=device, attn_implementation="sdpa")
model = PeftModel.from_pretrained(base, CKPT).merge_and_unload().eval()
print("로드 완료:", CKPT)

ALL_PERMS = [list(p) for p in itertools.permutations([1, 2, 3, 4])]
tok = processor.tokenizer
cand_ids = [tok.encode(format_chrono(p) + "<|im_end|>", add_special_tokens=False)
            for p in ALL_PERMS]
assert len({len(c) for c in cand_ids}) == 1, "후보 토큰 길이가 다름 — 토크나이저 확인"
CAND = torch.tensor(cand_ids)
print("candidate token length:", CAND.shape)

def build_messages(sentence, images):
    """기존 build_messages와 동일 구조 (Image N 인터리브)."""
    content = []
    for i, im in enumerate(images):
        content.append({"type": "image", "image": im})
        content.append({"type": "text", "text": f"\nImage {i + 1}\n"})
    content.append({"type": "text", "text": USER_TEXT_TMPL.format(sentence=sentence)})
    return [{"role": "user", "content": content}]

@torch.no_grad()
def score_view(sentence, frame_paths):
    imgs = [Image.open(p).convert("RGB") for p in frame_paths]
    inputs = processor.apply_chat_template(
        build_messages(sentence, imgs), tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt").to(device)

    out = model(**inputs, use_cache=True)
    last_logit = out.logits[:, -1:, :]
    cache = out.past_key_values
    cache.batch_repeat_interleave(24)
    cand = CAND.to(device)
    out2 = model(input_ids=cand, past_key_values=cache, use_cache=False)
    logits = torch.cat([last_logit.expand(24, -1, -1), out2.logits[:, :-1, :]], dim=1)
    logp = torch.log_softmax(logits.float(), dim=-1)
    tok_lp = logp.gather(-1, cand.unsqueeze(-1)).squeeze(-1)
    return tok_lp.sum(-1).cpu()          # [24] — 뷰 좌표 chrono 후보의 LL

@torch.no_grad()
def predict(sample, k_tta=None, seed=0):
    """원본 좌표의 chrono 예측 반환."""
    k_tta = k_tta or CFG["K_TTA"]
    rng = random.Random(zlib.crc32(str(sample["id"]).encode()) + seed)
    agg = {}
    for sigma in sample_perms(k_tta, rng):
        frames, _ = apply_perm(sample, sigma)
        scores = score_view(sample["caption"], frames)
        for ci, cand_view in enumerate(ALL_PERMS):
            cand_orig = tuple(sigma[i - 1] for i in cand_view)
            agg[cand_orig] = agg.get(cand_orig, 0.0) + scores[ci].item()
    return list(max(agg, key=agg.get))


체크포인트에 processor 없음, base 사용: Unrecognized processing class in /content/drive/MyDrive/snuai_runs_inprogress/qwen3vl8b_v5_r64_paug4_px320/checkpoint-2265. Can't instantiate a processor, a tokenizer, an image processor, a video processor or a feature extractor for this model. Make sure the repository contains the files of at least one of those processing classes.


Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

로드 완료: /content/drive/MyDrive/snuai_runs_inprogress/qwen3vl8b_v5_r64_paug4_px320/checkpoint-2265
candidate token length: torch.Size([24, 13])
